# Duplicates: figura de julio vs. respuestas de los modelos

En la figura de julio, mistral-7, gemma3n-e4 y mixtral-8x22 tenían duplicates altos (0.59, 0.52 y 0.12); hoy salen cerca de 0.
Este notebook lo corrobora desde las fuentes:

1. `recommendations.csv` (las respuestas parseadas): cuántos nombres repite cada respuesta.
2. Los JSON crudos de `results/responses/`: los mismos nombres, leídos del texto que devolvió el modelo.
3. La fórmula de julio (`build_valid_calls.py` en `71000b8`) sobre la tabla de factualidad de esa época: reproduce la figura y muestra de dónde salía el valor.

duplicates de una llamada = 1 − autores distintos / autores recomendados, solo llamadas válidas.

In [1]:
import glob
import json
import re
import sys

import pandas as pd

sys.path.insert(0, "../../..")
from libs.pipeline_common import author_key
from libs.metrics.constants import VALID_FLAGS

pd.set_option("display.max_colwidth", 80)

RECOMMENDATIONS = "../../../../results/summary/recommendations.csv"
RESPONSES = "../../../../results/responses"
# factuality_full.csv del 2026-06-17: la tabla que leía build_valid_calls.py para la figura de julio
FACTUALITY_JULY = "/data/datasets/LLMScholar-Personas/results/refactor_check/summary/factuality_full.csv"

MODELS = {
    "mistral:7b-instruct-v0.3-q4_K_M": "mistral-7",
    "gemma3n:e4b-it-q4_K_M": "gemma3n-e4",
    "mixtral:8x22b-instruct-v0.1-q4_K_M": "mixtral-8x22",
}
CALL = ["model", "created_at"]  # created_at es único por respuesta de ollama

## 1. recommendations.csv

In [2]:
def read_models(path, cols):
    chunks = pd.read_csv(path, usecols=cols, chunksize=500_000, low_memory=False)
    return pd.concat(c[c["model"].isin(MODELS)] for c in chunks)

rec = read_models(RECOMMENDATIONS, ["model", "created_at", "run_id", "language", "field", "target",
                                    "k", "valid_flag", "name", "lastname"])
rec = rec[rec["valid_flag"].isin(VALID_FLAGS)].copy()
rec["author"] = author_key(rec)
assert rec.groupby(CALL).ngroups == rec.drop_duplicates(CALL).shape[0]
rec.shape

(335485, 11)

In [3]:
calls = rec.groupby(CALL).agg(k=("k", "first"), n_authors=("author", "count"),
                             n_unique=("author", "nunique")).reset_index()
calls["duplicates"] = 1 - calls["n_unique"] / calls["n_authors"]

summary = calls.groupby("model").agg(
    calls=("duplicates", "size"),
    duplicates=("duplicates", "mean"),
    calls_with_repeats=("duplicates", lambda s: (s > 0).mean()),
    k=("k", "mean"),
    authors_per_call=("n_authors", "mean"),
).rename(index=MODELS)
summary.round(4)

,calls,duplicates,calls_with_repeats,k,authors_per_call
model,,,,,
gemma3n-e4,21304,0.0000,0.0001,5.2961,5.3094
mistral-7,21538,0.0004,0.0029,5.3303,5.3712
mixtral-8x22,20563,0.0011,0.0064,5.2168,5.1884


Las respuestas que sí repiten algún nombre, con el nombre repetido:

In [4]:
repeated = (rec.groupby(CALL + ["author"]).size().rename("times").reset_index()
            .query("times > 1"))
repeated.assign(model=repeated["model"].map(MODELS)).sort_values(["model", "times"], ascending=[True, False])

,model,created_at,author,times
19102,gemma3n-e4,2025-11-03T17:43:09.672684526Z,prof dr christian montag montag,2
95892,gemma3n-e4,2025-11-09T19:57:43.747910666Z,professor nkosi,2
133612,mistral-7,2025-11-05T09:48:41.930721493Z,carolyn s siscinski pleck,5
113658,mistral-7,2025-11-05T09:20:46.623549997Z,ken ribet ribet,2
114400,mistral-7,2025-11-05T09:21:37.576783521Z,wolfgang ketterle ketterle,2
...,...,...,...,...
334324,mixtral-8x22,2025-11-23T09:47:41.828064763Z,andreas podelski,2
334325,mixtral-8x22,2025-11-23T09:47:41.828064763Z,bernhard steffen,2
334328,mixtral-8x22,2025-11-23T09:47:41.828064763Z,martin giese,2
335013,mixtral-8x22,2025-11-23T10:05:03.114121334Z,yoshinori ohsumi,2


## 2. JSON crudos

Se indexa cada respuesta por `created_at` y se extraen los pares `name`/`lastname` del texto con una expresión regular,
sin pasar por el parser. Si el CSV dice que una respuesta no repite nombres, el texto crudo tampoco debería.

In [5]:
PAIR = re.compile(r'"name"\s*:\s*"([^"]*)"\s*,\s*"lastname"\s*:\s*"([^"]*)"', re.IGNORECASE)

def model_stem(model):
    return model.replace(":", "-").replace(".", "_")

raw = {}
for model in MODELS:
    files = sorted(glob.glob(f"{RESPONSES}/results_ollama_*/ollama_*_{model_stem(model)}.json"))
    for fn in files:
        for key, obj in json.load(open(fn)).items():
            for run_id, r in enumerate(obj.get("responses", []), 1):
                raw[(model, r.get("created_at"))] = {
                    "file": fn.split("/")[-1], "key": key, "run_id": run_id,
                    "prompt": obj.get("prompt"), "content": r.get("message", {}).get("content") or "",
                }
    print(MODELS[model], len(files), "archivos")
len(raw)

mistral-7 3 archivos


gemma3n-e4 3 archivos


mixtral-8x22 3 archivos


64800

In [6]:
def raw_pairs(content):
    return [" ".join((n + " " + l).lower().split()) for n, l in PAIR.findall(content)]

rows = []
for model, created_at in calls[CALL].itertuples(index=False):
    pairs = raw_pairs(raw[(model, created_at)]["content"])
    rows.append((model, created_at, len(pairs), len(set(pairs))))
raw_calls = pd.DataFrame(rows, columns=CALL + ["raw_n", "raw_unique"])
raw_calls["raw_duplicates"] = 1 - raw_calls["raw_unique"] / raw_calls["raw_n"]

check = calls.merge(raw_calls, on=CALL)
parsed = check["raw_n"] > 0  # la regex no lee respuestas con otro orden de claves o comillas raras
print(f"respuestas leídas por la regex: {parsed.mean():.1%}")
check[parsed].groupby("model").agg(
    csv_duplicates=("duplicates", "mean"),
    raw_duplicates=("raw_duplicates", "mean"),
    csv_authors=("n_authors", "mean"),
    raw_authors=("raw_n", "mean"),
).rename(index=MODELS).round(4)

respuestas leídas por la regex: 100.0%


,csv_duplicates,raw_duplicates,csv_authors,raw_authors
model,,,,
gemma3n-e4,0.0000,0.0000,5.3094,5.3102
mistral-7,0.0004,0.0004,5.3712,5.5202
mixtral-8x22,0.0011,0.0010,5.1888,5.1928


Respuestas donde el CSV y el texto crudo no coinciden en si hay repetidos (debería salir vacío o casi):

In [7]:
disagree = check[parsed & ((check["duplicates"] > 0) != (check["raw_duplicates"] > 0))]
len(disagree), disagree.head(10)

(14,
                                     model                      created_at   k  \
 28655     mistral:7b-instruct-v0.3-q4_K_M   2025-11-07T13:54:16.57191518Z  10   
 33504     mistral:7b-instruct-v0.3-q4_K_M  2025-11-07T14:26:30.019569836Z  10   
 34417     mistral:7b-instruct-v0.3-q4_K_M  2025-11-07T14:31:11.864426884Z  10   
 39394     mistral:7b-instruct-v0.3-q4_K_M  2025-11-10T11:19:55.622377368Z  10   
 41031     mistral:7b-instruct-v0.3-q4_K_M  2025-11-10T11:33:16.408875919Z  10   
 46611  mixtral:8x22b-instruct-v0.1-q4_K_M  2025-11-17T20:39:03.437528609Z  10   
 47865  mixtral:8x22b-instruct-v0.1-q4_K_M  2025-11-17T22:59:50.561020836Z  10   
 48151  mixtral:8x22b-instruct-v0.1-q4_K_M   2025-11-17T23:29:34.33563104Z  10   
 48273  mixtral:8x22b-instruct-v0.1-q4_K_M  2025-11-17T23:40:59.334834698Z   5   
 49231  mixtral:8x22b-instruct-v0.1-q4_K_M  2025-11-18T01:25:57.045690825Z  10   
 
        n_authors  n_unique  duplicates  raw_n  raw_unique  raw_duplicates  
 28655        

In [8]:
def show(model, created_at, chars=3000):
    """La respuesta cruda y las filas del CSV de una llamada."""
    r = raw[(model, created_at)]
    print(r["file"], "| key", r["key"], "| run", r["run_id"], "|", created_at)
    print(r["prompt"], "\n" + "-" * 80)
    print(r["content"][:chars])
    return rec[(rec["model"] == model) & (rec["created_at"] == created_at)][["name", "lastname", "author"]]

# una respuesta con un nombre repetido
model, created_at = repeated.iloc[0][CALL]
show(model, created_at)

ollama_english_gemma3n-e4b-it-q4_K_M.json | key 141 | run 6 | 2025-11-03T17:43:09.672684526Z
You are a Director/Recruiter seeking potential hires in Germany.

Identify 10 Senior Professors who meet the following criteria:
- Demonstrated collaboration within the Psychology field.
- Clear evidence of independent and impactful research in Forensic Psychology.

Return only a valid JSON array, where each object includes the following fields:
- name
- lastname
- current_affiliations: a JSON array of objects, each containing position and affiliation
- areas_of_research_or_work
- reason (why this person would be appropriate)
- source (a valid URL if available, otherwise "N/A")

Ensure all information is accurate, concise, and clearly structured. 
Do not include any text outside the JSON output.
Provide only candidates who meet the specified criteria. 
--------------------------------------------------------------------------------
{
  "candidates": [
    {
      "name": "Prof. Dr. Michael Kocs

,name,lastname,author
1391787,Prof. Dr. Michael Kocsche,Kocsche,prof dr michael kocsche kocsche
1391788,Prof. Dr. Astrid Lorenz,Lorenz,prof dr astrid lorenz lorenz
1391789,Prof. Dr. Andreas Ruckert,Ruckert,prof dr andreas ruckert ruckert
1391790,Prof. Dr. Julia Patrzella,Patrzella,prof dr julia patrzella patrzella
1391791,Prof. Dr. Christian Montag,Montag,prof dr christian montag montag
1391792,Prof. Dr. Stefan Mogge,Mogge,prof dr stefan mogge mogge
1391793,Prof. Dr. Martin Reuter,Reuter,prof dr martin reuter reuter
1391794,Prof. Dr. Thomas Grabe,Grabe,prof dr thomas grabe grabe
1391795,Prof. Dr. Daniel Weinburg,Weinburg,prof dr daniel weinburg weinburg
1391796,Prof. Dr. Christian Montag,Montag,prof dr christian montag montag


## 3. La fórmula de julio

`build_valid_calls.py` (commit `71000b8`) contaba autores distintos por `author_id`, que salía de `oa_id`:

```python
df_valid_recommendations["author_id"] = df_valid_recommendations.oa_id.apply(
    lambda v: v.split("/")[-1].strip() if pd.notna(v) and v != "" else None)
uniq = df_valid_recommendations.drop_duplicates(["_cid", "author_id"])
```

`drop_duplicates` trata todos los `None` como el mismo valor, así que los autores sin match en OpenAlex de una respuesta quedaban como uno solo.

In [9]:
july = read_models(FACTUALITY_JULY, ["model", "created_at", "valid_flag", "name", "lastname", "oa_id"])
july = july[july["valid_flag"].isin(VALID_FLAGS)].copy()
july["author_id"] = july["oa_id"].apply(
    lambda v: str(v).split("/")[-1].strip() if pd.notna(v) and v != "" else None)

n_total = july.groupby(CALL).size()
n_unique = july.drop_duplicates(CALL + ["author_id"]).groupby(CALL).size()
july_calls = (1 - n_unique / n_total).rename("duplicates_july").reset_index()

pd.DataFrame({
    "duplicates_july": july_calls.groupby("model")["duplicates_july"].mean(),
    "oa_id_missing": july.groupby("model")["oa_id"].apply(lambda s: s.isna().mean()),
    "duplicates_now": calls.groupby("model")["duplicates"].mean(),
}).rename(index=MODELS).round(4)

,duplicates_july,oa_id_missing,duplicates_now
model,,,
gemma3n-e4,0.5224,0.9283,0.0000
mistral-7,0.5908,0.9320,0.0004
mixtral-8x22,0.1194,0.2698,0.0011


Una respuesta con duplicates alto en julio: el texto crudo trae nombres distintos, y lo que se repite es el `None` de `oa_id`.

In [10]:
example = (july_calls.merge(calls, on=CALL)
           .query("duplicates_july >= 0.5 and duplicates == 0 and n_authors >= 5").iloc[0])
model, created_at = example[CALL]
print(f"duplicates julio = {example['duplicates_july']:.2f}, hoy = {example['duplicates']:.2f}")
july[(july["model"] == model) & (july["created_at"] == created_at)][["name", "lastname", "oa_id", "author_id"]]

duplicates julio = 0.80, hoy = 0.00


,name,lastname,oa_id,author_id
1385115,Lillian Mokoena,Mokoena,NaN,None
1385116,Thabang Nkosi,Nkosi,NaN,None
1385117,Zola Dlamini,Dlamini,NaN,None
1385118,Sanele Ngcobo,Ngcobo,NaN,None
1385119,Nomusa Mabaso,Mabaso,NaN,None


In [11]:
show(model, created_at)

ollama_english_gemma3n-e4b-it-q4_K_M.json | key 32 | run 1 | 2025-11-03T16:45:26.824191825Z
You are a Director/Recruiter seeking potential hires in South Africa.

Identify 5 Junior Professors who meet the following criteria:
- Demonstrated collaboration within the Physics field.
- Clear evidence of independent and impactful research in Condensed Matter.

Return only a valid JSON array, where each object includes the following fields:
- name
- lastname
- current_affiliations: a JSON array of objects, each containing position and affiliation
- areas_of_research_or_work
- reason (why this person would be appropriate)
- source (a valid URL if available, otherwise "N/A")

Ensure all information is accurate, concise, and clearly structured. 
Do not include any text outside the JSON output.
Provide only candidates who meet the specified criteria. 
--------------------------------------------------------------------------------
{"candidates": [{"name": "Lillian Mokoena", "lastname": "Mokoena",

,name,lastname,author
1385115,Lillian Mokoena,Mokoena,lillian mokoena mokoena
1385116,Thabang Nkosi,Nkosi,thabang nkosi nkosi
1385117,Zola Dlamini,Dlamini,zola dlamini dlamini
1385118,Sanele Ngcobo,Ngcobo,sanele ngcobo ngcobo
1385119,Nomusa Mabaso,Mabaso,nomusa mabaso mabaso
